In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess

from utils import compose_windows, make_names_dict

import statsmodels.api as sm

from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, accuracy_score, roc_auc_score

## Dobra wersja wykorzystująca `statsmodels` (20.01.26)

## TODO

Zrobić to w statsmodels, tak jak opisano na Mattermost. W statsmodels trzeba ręczenie dodać wyraz wolny w modelu liniowym.

Dodać do df informacje o wagach i zapisać to w postaci long

Narysować współczynninki z regresji logistycznej z przedziałami ufności

Sprawdzić czemu w obecnych wynikach wszystkie wiersze wychodzą identycznie

### Logistic Regression regularization trajecotries:

Here we only investigate what happens to the model weights as we turn up the L2. L1 and ElasticNet are ignored only for simplicity and no other reason.

For each tissue, a DataFrame is created that contains:

2. Lists of most important motifs based on weights of the model **trained on all of the data**

In [2]:
windows=["06-08", "10-12", "14-16"]
tissues=["Neuroblasts", "Neurons", "Glia"]
model_name = "Logistic Regression"
model_short = "LR"

n_splits = 10   # cross-validation splits

alpha_values = np.array([0, 0.01, 0.1, 0.2, 0.5, 1, 2, 10]) # values for the C parameter in LogisticRegression()
                                                          # C is the INVERSE of regularization strength

In [11]:
def reg_trajectory_statsmodels(
    tissue: str,
    alpha_values: np.array,
    n_splits: int = 10,
    top_n: int = 20,
    motif_annotations_path: str = "data/motif_names.tsv"
) -> pd.DataFrame:

    # Load windows
    X, y, composite = compose_windows(tissue)

    results = []

    for alpha in alpha_values:

        # Load motif annotations
        annot = pd.read_csv(motif_annotations_path, sep="\t")

        code_to_name = (
            annot
            .drop_duplicates("id")
            .set_index("id")["name"]
            .astype(str)
            .to_dict()
        )

        mapped_columns = (
            X.columns
            .to_series()
            .map(code_to_name)
            .fillna(pd.Series(X.columns, index=X.columns))
        )

        X_named = X.copy()
        X_named.columns = mapped_columns

        # Final fit on full data
        X_full_sm = sm.add_constant(X_named)
        model_full = sm.Logit(y, X_full_sm)

        # Regularized fit for weights
        if alpha == 0.0:
            res_reg = model_full.fit(disp=False)
        else:
            res_reg = model_full.fit_regularized(
                alpha=alpha,
                L1_wt=0.0,
                disp=False
            )

        # Unregularized fit for inference (CIs)
        try:
            res_inf = model_full.fit(disp=False)
        except np.linalg.LinAlgError:       # in case of a Singular matrix showing up
            res_inf = model_full.fit_regularized(
                alpha=1e-6,
                L1_wt=0.0,
                disp=False
            )

        probs = res_reg.predict(X_full_sm)
        preds = (probs > 0.5).astype(int)

        roc_auc = roc_auc_score(y, probs)
        acc = accuracy_score(y, preds)

        params = res_reg.params
        conf_int = res_inf.conf_int()
        conf_int.columns = ["ci_lower", "ci_upper"]

        coef_df = (
            pd.concat([params, conf_int], axis=1)
            .reset_index()
            .rename(columns={"index": "feature", 0: "weight"})
        )

        # Drop intercept from ranking
        coef_df = coef_df[coef_df["feature"] != "const"]

        coef_df["abs_weight"] = coef_df["weight"].abs()
        coef_df = coef_df.sort_values("abs_weight", ascending=False)

        top_df = coef_df.head(top_n)

        mean_abs_top_weights = top_df["abs_weight"].mean()

        # Store results
        row = {
            "mean abs(top weights)": mean_abs_top_weights,
            "AUC ROC": roc_auc,
            "Accuracy": acc,
        }

        for i, (_, r) in enumerate(top_df.iterrows(), start=1):
            row[f"motif {i}"]      = r["feature"]
            row[f"motif {i} β"]    = r["weight"]
            row[f"motif {i} CI lo"] = r["ci_lower"]
            row[f"motif {i} CI hi"] = r["ci_upper"]

        results.append(row)

    trajectory = pd.DataFrame(results, index=alpha_values)
    trajectory.index.name = "alpha parameter value"
 
    dir = "results/data/regularization_trajectories/statsmodels"
    os.makedirs(dir, exist_ok=True)
    path = os.path.join(dir, f"{tissue}_top_motifs.tsv")
    trajectory.to_csv(path, sep="\t")

    return trajectory


In [10]:
tissue = "Neuroblasts"

X, y, composite = compose_windows(tissue)
# X.head()

np.std(X, axis=0).describe()

count    361.000000
mean       0.186989
std        0.132707
min        0.047878
25%        0.101459
50%        0.150244
75%        0.222643
max        1.092077
dtype: float64

In [ ]:
full_rank = np.linalg.matrix_rank(X)
colinear_names = []

for col_name in X.columns:

    X_curr = X.drop(columns=X.columns[col_idx])
    # print(X_curr.shape)
    # pd.concat((X.iloc[:,:col_idx], X.iloc[:,col_idx:]), axis=1)
    
    curr_rank = np.linalg.matrix_rank(X_curr)
    if curr_rank == full_rank:
        X
        colinear_names.append(X_curr.columns[col_idx])

colinear_names

['M5280-1.02',
 'M5222-1.02',
 'M5203-1.02',
 'M5194-1.02',
 'M5179-1.02',
 'M5174-1.02',
 'M5173-1.02',
 'M5142-1.02',
 'M5111-1.02',
 'M5109-1.02',
 'M5108-1.02',
 'M5079-1.02',
 'M5065-1.02',
 'M5052-1.02',
 'M5051-1.02',
 'M5035-1.02',
 'M5029-1.02',
 'M5009-1.02',
 'M4931-1.02',
 'M4930-1.02',
 'M4921-1.02',
 'M4919-1.02',
 'M4897-1.02',
 'M4896-1.02',
 'M4895-1.02',
 'M4894-1.02',
 'M4808-1.02',
 'M4747-1.02',
 'M4745-1.02',
 'M4735-1.02',
 'M4728-1.02',
 'M4725-1.02',
 'M2259-1.02',
 'M2060-1.02',
 'M2059-1.02',
 'M2058-1.02',
 'M2057-1.02',
 'M2049-1.02',
 'M2039-1.02',
 'M2038-1.02',
 'M2037-1.02',
 'M2031-1.02',
 'M2026-1.02',
 'M2021-1.02',
 'M2018-1.02',
 'M2013-1.02',
 'M2009-1.02',
 'M2007-1.02',
 'M2006-1.02',
 'M2002-1.02',
 'M1995-1.02',
 'M1992-1.02',
 'M1986-1.02',
 'M1980-1.02',
 'M1978-1.02',
 'M1977-1.02',
 'M0737-1.02']

In [46]:
len(colinear_names)

57

In [38]:
X_curr.shape

NameError: name 'X_curr' is not defined

In [ ]:
X.iloc[:,:12]

,M0111-1.02,M0135-1.02,M0302-1.02,M0319-1.02,M0576-1.02,M0580-1.02,M0596-1.02,M0641-1.02,M0713-1.02,M0723-1.02,M0736-1.02,M0737-1.02
L1,0.036434,-0.042761,-0.055937,-0.051939,0.155917,0.120051,0.073466,0.143631,0.113457,-0.087612,-0.128754,-0.089362
L2,0.013458,-0.014428,-0.100512,-0.101407,0.168278,0.100617,0.124362,0.165200,0.211817,-0.217889,-0.289336,-0.158100
L3,-0.082908,-0.063233,-0.037911,-0.054506,0.183282,0.134704,0.072459,0.110677,0.138229,-0.098284,-0.133884,-0.034282
L4,-0.064761,-0.126266,-0.081522,-0.237871,0.184288,0.379370,0.149114,0.160640,0.568599,-0.205610,-0.121015,-0.159113
L5,-0.058579,-0.108866,-0.096683,-0.168254,0.134865,0.054117,0.028358,0.153212,0.263527,-0.335852,-0.368674,-0.255424
...,...,...,...,...,...,...,...,...,...,...,...,...
L413,0.059237,0.200967,0.059811,0.073849,0.315860,0.279247,0.183642,0.145484,0.208913,0.047320,0.076041,0.003569
L414,-0.010428,0.040537,0.118873,0.135509,0.391476,0.301709,0.081438,-0.001165,0.191609,0.319228,0.311998,0.246226
L415,-0.045608,0.409092,-0.016070,-0.016976,0.317745,0.188064,0.190938,0.269595,0.598039,0.068187,0.032673,0.096325
L416,0.031320,0.349451,0.093721,0.124274,0.252142,0.148273,0.170290,0.317841,0.584092,0.134644,0.095258,0.186395


In [33]:
pd.concat((X.iloc[:,:12], X.iloc[:,12:]), axis=1)

,M0111-1.02,M0135-1.02,M0302-1.02,M0319-1.02,M0576-1.02,M0580-1.02,M0596-1.02,M0641-1.02,M0713-1.02,M0723-1.02,...,M6235-1.02,M6237-1.02,M6238-1.02,M6277-1.02,M6374-1.02,M6394-1.02,M6410-1.02,M6459-1.02,M6501-1.02,M6537-1.02
L1,0.036434,-0.042761,-0.055937,-0.051939,0.155917,0.120051,0.073466,0.143631,0.113457,-0.087612,...,-0.091355,-0.207948,-0.216468,0.115733,-0.074520,0.093541,-0.012944,-0.065907,0.189042,0.255387
L2,0.013458,-0.014428,-0.100512,-0.101407,0.168278,0.100617,0.124362,0.165200,0.211817,-0.217889,...,-0.186711,-0.330159,-0.321703,0.084260,-0.041580,0.091467,-0.051219,-0.117627,0.246027,0.285014
L3,-0.082908,-0.063233,-0.037911,-0.054506,0.183282,0.134704,0.072459,0.110677,0.138229,-0.098284,...,-0.114736,-0.312655,-0.235169,-0.044793,0.007528,0.015937,-0.012050,-0.205472,0.234470,0.215652
L4,-0.064761,-0.126266,-0.081522,-0.237871,0.184288,0.379370,0.149114,0.160640,0.568599,-0.205610,...,-0.173884,-0.090732,-0.203616,-0.001836,0.014685,0.318478,0.096175,0.131480,0.158498,0.469570
L5,-0.058579,-0.108866,-0.096683,-0.168254,0.134865,0.054117,0.028358,0.153212,0.263527,-0.335852,...,-0.274995,-0.439190,-0.395376,0.030479,-0.067681,0.047614,-0.032654,-0.131073,0.209884,0.202136
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
L413,0.059237,0.200967,0.059811,0.073849,0.315860,0.279247,0.183642,0.145484,0.208913,0.047320,...,-0.002746,0.052285,0.069886,0.146660,0.015111,0.223810,0.069349,-0.032172,0.298511,0.328556
L414,-0.010428,0.040537,0.118873,0.135509,0.391476,0.301709,0.081438,-0.001165,0.191609,0.319228,...,0.194222,-0.097041,0.063835,0.090195,-0.008257,0.243621,-0.007743,-0.310927,0.269007,0.324338
L415,-0.045608,0.409092,-0.016070,-0.016976,0.317745,0.188064,0.190938,0.269595,0.598039,0.068187,...,0.049982,-0.102087,0.048681,0.011269,0.088268,0.184967,0.131653,-0.080557,0.360226,0.397849
L416,0.031320,0.349451,0.093721,0.124274,0.252142,0.148273,0.170290,0.317841,0.584092,0.134644,...,0.113152,0.008257,0.092478,0.115074,0.003184,0.218756,0.143382,0.175405,0.322534,0.345653


In [22]:
X.shape[1]

361

In [19]:
np.unique(X, axis =0)

array([[-3.11686849e-01,  4.04342555e-01,  2.50007327e-01, ...,
         9.68904006e-04,  2.75349698e-01,  2.27786016e-01],
       [-2.70987454e-01,  5.40125523e-01,  1.56854962e-01, ...,
        -2.37903928e-01,  3.38262093e-01,  1.96714684e-01],
       [-2.05042941e-01, -6.11944517e-02,  1.66691041e-01, ...,
        -4.25073405e-01,  1.85205313e-01,  2.18333318e-01],
       ...,
       [ 6.61045718e-01,  6.82201862e-01,  1.78840866e+00, ...,
         4.02785678e-01,  4.32233818e-01,  5.11841237e-01],
       [ 6.87278434e-01,  6.52402953e-01,  4.76015958e-01, ...,
         4.30319538e-01,  2.22609454e-01,  4.88304402e-01],
       [ 7.48344005e-01,  5.42136124e-01,  1.05592289e+00, ...,
         5.31207227e-01,  6.48693999e-01,  5.21136661e-01]],
      shape=(1200, 361))

In [18]:
for t in tissues:
    _ = reg_trajectory_statsmodels(alpha_values=alpha_values, tissue=t)

/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/statsmodels/base/l1_solvers_common.py:71: ConvergenceWarning: QC check did not pass for 272 out of 362 parameters
Try increasing solver accuracy or number of iterations, decreasing alpha, or switch solvers
  warnings.warn(message, ConvergenceWarning)
/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/statsmodels/base/l1_solvers_common.py:144: ConvergenceWarning: Could not trim params automatically due to failed QC check. Trimming using trim_mode == 'size' will still work.
  warnings.warn(msg, ConvergenceWarning)


LinAlgError: Singular matrix

## Zła wersja wykorzystująca `sklearn` (13.01.26)

### Logistic Regression regularization trajecotries:

Here we only investigate what happens to the model weights as we turn up the L2. L1 and ElasticNet are ignored only for simplicity and no other reason.

For each tissue, a DataFrame is created that contains:
1. Metrics (mean and std of ACC and AUC ROC) computed on **CROSS-VALIDATION**
2. Lists of most important motifs based on weights of the model **trained on all of the data**

In [ ]:
windows=["06-08", "10-12", "14-16"]
tissues=["Neuroblasts", "Neurons", "Glia"]
model_name = "Logistic Regression"
model_short = "LR"

n_splits = 10   # cross-validation splits

c_values = np.array([np.inf, 100, 10, 5, 2, 1, 0.5, 0.1]) # values for the C parameter in LogisticRegression()
                                                          # C is the INVERSE of regularization strength

In [18]:

c_values = np.array([np.inf, 100, 10, 5, 2, 1, 0.5, 0.1]) # values for the C parameter in LogisticRegression()
                                                          # C is the INVERSE of regularization strength

def reg_trajectory(tissue: str, c_values: np.array(list[int]), n_splits: int = 10, top_n: int = 20, motif_annotations_path: str = "data/motif_names.tsv") -> pd.DataFrame :

    # Load windows
    X, y, composite = compose_windows(tissue)

    results = []

    for c in c_values:

        # -----------------------
        # Cross-validation
        # -----------------------
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
        probs, trues, accs = [], [], []

        base_classifier = LogisticRegression(
            penalty="l2",
            C = c,
            n_jobs=-1
        )

        for i, (train_idx, test_idx) in enumerate(skf.split(X, composite)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            classifier = clone(base_classifier)
            classifier.fit(X_train, y_train)

            p = classifier.predict_proba(X_test)[:, 1]
            probs.append(p)
            trues.append(y_test.values)
            accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

        # Accuracy metrics
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        # ROC AUC metrics
        roc_aucs = []
        for true, prob in zip(trues, probs):
            fpr, tpr, _ = roc_curve(true, prob)
            roc_aucs.append(auc(fpr, tpr))

        mean_auc = np.mean(roc_aucs)
        std_auc = np.std(roc_aucs)

        # Load annotations, then train on full dataset

        annot = pd.read_csv(motif_annotations_path, sep="\t")

        # Build mapping: motif code -> motif name
        code_to_name = (
            annot
            .drop_duplicates("id")
            .set_index("id")["name"]
            .astype(str)
            .to_dict()
        )

        # Map columns; keep original code if annotation is missing
        mapped_columns = (
            X.columns
            .to_series()
            .map(code_to_name)
            .fillna(pd.Series(X.columns, index=X.columns))
        )
        X = X.copy()
        X.columns = mapped_columns

        # train on full dataset
        full_classifier = clone(base_classifier)
        full_classifier.fit(X, y)

        weights = full_classifier.coef_.ravel()

        coef_df = pd.DataFrame({
            "feature": X.columns,
            "weight": weights
        })

        coef_df["abs_weight"] = coef_df["weight"].abs()
        coef_df = coef_df.sort_values("abs_weight", ascending=False)

        top_df = coef_df.head(top_n)

        top_motifs = top_df["feature"].tolist()
        mean_abs_top_weights = top_df["abs_weight"].mean()

        # Store results
        row = {
            "mean abs(top weights)": mean_abs_top_weights,
            "mean AUC ROC": mean_auc,
            "std AUC ROC": std_auc,
            "mean Accuracy": mean_acc,
            "std Accuracy": std_acc
        }

        # Add motif columns: motif 1, motif 2, ...
        for i, motif in enumerate(top_motifs, start=1):
            row[f"motif {i}"] = motif

        results.append(row)

    # Construct final dataframe
    trajectory = pd.DataFrame(results, index=c_values)
    trajectory.index.name = "c parameter value"
    dir = "results/data/regularization_trajectories/sklearn_prototype"
    os.makedirs(dir, exist_ok=True)
    path = os.path.join(dir, f"{tissue}_top_motifs.tsv")
    trajectory.to_csv(path, sep="\t")

    return trajectory

In [19]:
for t in tissues:
    _ = reg_trajectory(c_values=c_values, tissue=t)

/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/olek/miniconda3/envs/env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules

KeyboardInterrupt: 